# Pipeline de Decisión — Caso 3

**Notebook de desarrollo interactivo:** contiene el código fuente del pipeline inline,
permite probar casos individuales o en lote, con flags para controlar sobreescritura
y persistencia en BD.

## Flujo del pipeline

```
load_case → compute_features → apply_rules
                                    │
                        ┌───────────┼───────────┐
                        ▼           ▼           ▼
                    RECHAZAR    APROBAR     ambiguo
                        │           │           │
                        ▼           ▼     llm_classify
                   final_decision   │           │
                        │           │     final_decision
                        ▼           ▼           │
                   generate_output ◄────────────┘
                        │
                       END
```

## Flags configurables

| Flag | Default | Descripción |
|------|---------|-------------|
| `OVERWRITE` | `False` | Si `True`, reprocesa casos ya analizados |
| `SAVE_TO_DB` | `True` | Si `True`, persiste resultados en PostgreSQL |
| `USE_DB_RULES` | `True` | Si `True`, usa reglas de DB; si `False`, YAML |
| `SHOW_LLM_DETAIL` | `True` | Mostrar análisis estructurado del LLM en output |

> **Diferencia con `07_pipeline_test.ipynb`:** esta notebook contiene el código
> fuente del pipeline inline (editable en caliente), controles de sobreescritura,
> y ejecución batch interactiva. La 07 es solo demo con resultados pre-computados.

In [13]:
import os
import sys
import json
import re
import time
from pathlib import Path
from typing import Any, TypedDict

from dotenv import load_dotenv

try:
    CWD = Path.cwd()
except OSError:
    CWD = Path(".").resolve()
PROYECTO = CWD.parent if CWD.name == "notebooks" else CWD
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
try:
    os.chdir(str(PROYECTO))
except OSError:
    pass

load_dotenv(PROYECTO / ".env")

import pandas as pd
import psycopg2
import psycopg2.extras
import yaml
from langgraph.graph import END, StateGraph

# ── Flags interactivos (modificar aquí) ──────────────────────────
OVERWRITE = True        # True: reprocesa casos ya analizados
SAVE_TO_DB = True        # True: persiste en PostgreSQL
USE_DB_RULES = False  # reglas desde thresholds.yaml (sin DB)      # True: reglas de DB; False: YAML fallback
SHOW_LLM_DETAIL = True   # mostrar análisis LLM en output

# ── Conexión DB ─────────────────────────────────────────────────
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '5432')),
    'dbname': os.getenv('DB_NAME', 'rappi_cases'),
    'user': os.getenv('DB_USER', 'rappi'),
    'password': os.getenv('DB_PASSWORD', 'rappi_pass'),
}

def db_conectar():
    return psycopg2.connect(**DB_CONFIG)

# Verificar conexión
try:
    conn = db_conectar()
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM cases")
        total_casos = cur.fetchone()[0]
    conn.close()
    print(f'[OK] Setup completo — {total_casos} casos en DB')
except Exception as e:
    print(f'[WARN] DB no disponible: {e}')
    print('[INFO] SAVE_TO_DB se forzará a False')
    SAVE_TO_DB = False

[OK] Setup completo — 250 casos en DB


## Pipeline: State + Nodos 1-2 (load_case, compute_features)

El estado `CaseState` viaja entre nodos del grafo. `load_case` carga desde
PostgreSQL; `compute_features` calcula las 16 features o usa las precalculadas.

In [14]:
# ── State ─────────────────────────────────────────────────────────
class CaseState(TypedDict, total=False):
    case_id: str
    raw_data: dict[str, Any]
    features: dict[str, Any]
    features_version: str
    rule_result: str | None
    decision_regla: str | None           # APROBAR | RECHAZAR | AMBIGUO | ESCALAR
    decision_llm: str | None             # veredicto discreto del LLM
    justificacion_regla: str             # un bloque por regla disparada (multilínea)
    justificacion_llm: str | None
    senales_regla: list[str]             # "campo operador umbral = valor real"
    senales_llm: list[str]
    rule_details: list[dict[str, Any]]
    reglas_checklist: list[dict[str, Any]]
    llm_analysis: dict[str, Any] | None
    llm_resultado: dict[str, Any] | None
    final_decision: str
    justification: str                   # derivado para API/UI
    rule_disparada: str
    es_sintetico: bool

# ── Nodo 1: load_case ────────────────────────────────────────────
def load_case(state: CaseState) -> CaseState:
    """Carga los datos crudos del caso desde PostgreSQL (solo columnas de ``cases``)."""
    conn = db_conectar()
    try:
        with conn.cursor() as cur:
            cur.execute(
                """SELECT caso_id, usuario_id, antiguedad_usuario_dias, ciudad,
                          vertical, restaurante, valor_orden_mxn,
                          compensacion_solicitada_mxn, num_compensaciones_90d,
                          monto_compensado_90d_mxn, entrega_confirmada_gps,
                          tiempo_entrega_real_min, flags_fraude_previos,
                          motivo_reclamo, descripcion_reclamo,
                          recomendacion_agente, es_sintetico
                   FROM cases WHERE caso_id = %s""",
                (state["case_id"],),
            )
            row = cur.fetchone()
            if row is None:
                raise ValueError(f"Caso {state['case_id']} no encontrado")
            cols = [desc[0] for desc in cur.description]
            raw = dict(zip(cols, row))
    finally:
        conn.close()
    state["raw_data"] = raw
    state["es_sintetico"] = bool(raw.get("es_sintetico", False))
    return state

# ── Nodo 2: compute_features ─────────────────────────────────────
from src.pipeline.features import (
    calcular_features,
    cargar_features,
    persistir_features,
)

VERSION_FEATURES = "v1"

def compute_features(state: CaseState) -> CaseState:
    """Carga (o calcula y persiste) las features del caso.

    - Si el caso ya tiene features en la tabla ``features`` → las reutiliza.
    - Si no → las calcula de forma determinista desde ``raw_data`` y las
      persiste (upsert) para no reprocesar.
    ``features.version`` se copia a ``features_version`` (auditoría en
    ``resolution_case.features_version``).
    """
    raw = state["raw_data"]
    case_id = state["case_id"]
    conn = db_conectar()
    try:
        existentes = cargar_features(conn, case_id)
        if existentes is not None:
            features = {k: v for k, v in existentes.items() if k != "version"}
            state["features_version"] = existentes.get("version", VERSION_FEATURES)
        else:
            features = calcular_features(raw)
            persistir_features(conn, case_id, features, VERSION_FEATURES)
            conn.commit()
            state["features_version"] = VERSION_FEATURES
    finally:
        conn.close()
    # El motor de reglas evalúa sobre raw + features combinados.
    state["features"] = {**raw, **features}
    return state

print('[OK] State + load_case + compute_features definidos')

[OK] State + load_case + compute_features definidos


## Pipeline: Nodo 3 — apply_rules

Evalúa el caso contra las reglas de `src/rules/thresholds.yaml` (motor
`RuleEngine`). No hay reglas gestionadas en base de datos.

**Precedencia:** ESCALAR_FORZOSO → RECHAZAR → APROBAR → AMBIGUO

In [15]:
from src.rules.rule_engine import RuleEngine

def apply_rules(state: CaseState) -> CaseState:
    """Aplica las reglas del YAML (thresholds.yaml) al caso."""
    features = state["features"]
    engine = RuleEngine()
    df = pd.DataFrame([features])
    row = engine.decide(df).iloc[0]

    decision = row["recomendacion"]
    senales = [s for s in str(row["senales_usadas"]).split(" | ") if s]

    state["justificacion_regla"] = row["justificacion"]
    state["rule_details"] = []
    state["rule_disparada"] = ""
    state["senales_regla"] = [s for s in senales if "ambiguo" not in s.lower()]

    if decision == "ESCALAR" and any("ambiguo" in s.lower() for s in senales):
        state["decision_regla"] = "AMBIGUO"
        state["rule_result"] = None
        state["senales_regla"] = []
    elif decision == "ESCALAR":
        state["decision_regla"] = "ESCALAR"
        state["rule_result"] = "ESCALAR"
    else:
        state["decision_regla"] = decision
        state["rule_result"] = decision
    return state

print("[OK] apply_rules definido (RuleEngine · thresholds.yaml)")


[OK] apply_rules definido (RuleEngine · thresholds.yaml)


## Pipeline: Nodo 4 — llm_classify

Analiza `descripcion_reclamo` con LLM (OpenRouter, modelo `src/config/model.yaml`) para casos
ambiguos. Produce un JSON con justificación, resumen, veredicto y señales explicadas.
El `veredicto` contiene la decisión directamente (`APROBAR: ...`, `RECHAZAR: ...`, `ESCALAR: ...`).

In [16]:
"""Nodo llm_classify: análisis LLM de descripcion_reclamo para casos ambiguos.

Usa un prompt estructurado con system prompt, 3 few-shot examples y contexto
completo de reglas evaluadas para maximizar determinismo (temperature=0).

Nodo asíncrono: el cuello de botella es I/O (la espera de red del LLM), por lo
que el event loop de FastAPI intercala las esperas de varios casos on-demand.
La concurrencia de llamadas simultáneas al LLM está limitada por
``LLM_SEMAFORO`` para respetar el rate-limit del proveedor (OpenRouter).
"""

import asyncio
import json
import logging
import os
import re

from langchain_openai import ChatOpenAI

from src.config import get_llm_config, load_prompts
from src.pipeline.state import CaseState
from src.rules.signals import normalizar_señal_llm

logger = logging.getLogger(__name__)

_SYSTEM_PROMPT, _EJEMPLOS = load_prompts()
# Se ensambla igual que el prompt histórico original (bloques separados por
# línea en blanco + salto de línea final) para no alterar el texto enviado.
SYSTEM_PROMPT = f"{_SYSTEM_PROMPT}\n\n{_EJEMPLOS}\n"

# Cliente compartido: una sola instancia reutiliza las conexiones httpx y evita
# el setup por llamada. max_retries reintenta automáticamente 429/5xx/timeout.
_llm: ChatOpenAI | None = None

# Límite de llamadas LLM simultáneas (protege el rate-limit del proveedor).
LLM_SEMAFORO = asyncio.Semaphore(get_llm_config().max_concurrencia)


def _crear_cliente() -> ChatOpenAI:
    """Devuelve el cliente ChatOpenAI (singleton a nivel módulo).

    Los parámetros (model, base_url, temperature, max_tokens, max_retries,
    timeout) se leen de ``src/config/model.yaml`` con override por env var.
    """
    global _llm
    if _llm is None:
        cfg = get_llm_config()
        _llm = ChatOpenAI(
            model=cfg.model,
            api_key=os.getenv("OPENROUTER_API_KEY"),
            base_url=cfg.base_url,
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
            max_retries=cfg.max_retries,
            timeout=cfg.timeout_seconds,
        )
        os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY", "")
    return _llm


def _formatear_reglas_evaluadas(rule_details: list[dict]) -> str:
    if not rule_details:
        return "N/A"
    partes = []
    for r in rule_details:
        status = "✓" if r.get("se_disparo") else "✗"
        partes.append(f"  {status} {r.get('regla_id', '?')}: {r.get('nombre', '?')}")
    return "\n".join(partes)


def _extraer_json(contenido: str) -> str:
    """Extrae el bloque JSON de la respuesta del modelo."""
    idx_inicio = contenido.find("{")
    idx_fin = contenido.rfind("}")
    if idx_inicio != -1 and idx_fin != -1 and idx_fin > idx_inicio:
        return contenido[idx_inicio:idx_fin + 1]
    if "```json" in contenido:
        return contenido.split("```json")[1].split("```")[0]
    if "```" in contenido:
        return contenido.split("```")[1].split("```")[0]
    return contenido


def _parsear_analisis(contenido: str) -> dict | None:
    """Interpreta la respuesta del modelo como JSON estructurado.

    Returns:
        Dict con ``justificacion``, ``resumen``, ``veredicto`` y
        ``señales_explicadas`` si la respuesta es interpretable; ``None`` si no.
    """
    contenido = contenido.strip()
    if not contenido:
        return None

    json_str = _extraer_json(contenido).strip()
    try:
        analisis = json.loads(json_str)
    except json.JSONDecodeError:
        just_m = re.search(r'"justificacion"\s*:\s*"((?:[^"\\]|\\.)*)', json_str)
        res_m = re.search(r'"resumen"\s*:\s*"((?:[^"\\]|\\.)*)', json_str)
        ver_m = re.search(r'"veredicto"\s*:\s*"((?:[^"\\]|\\.)*)', json_str)
        if not ver_m:
            return None
        analisis = {
            "justificacion": just_m.group(1) if just_m else "",
            "resumen": res_m.group(1) if res_m else "",
            "veredicto": ver_m.group(1),
            "señales_explicadas": [],
        }

    if not isinstance(analisis, dict):
        return None
    if not all(k in analisis for k in ("justificacion", "resumen", "veredicto")):
        return None
    if not str(analisis.get("veredicto", "")).strip():
        return None
    return analisis


def _veredicto_discreto(veredicto: str) -> str:
    """Extrae el resultado discreto (APROBAR | RECHAZAR | ESCALAR) del veredicto.

    Args:
        veredicto: Texto del veredicto del LLM (ej. "ESCALAR: ambigüedad...").

    Returns:
        Decisión discreta; ESCALAR si no se puede interpretar.
    """
    upper = (veredicto or "").strip().upper()
    if upper.startswith("RECHAZAR"):
        return "RECHAZAR"
    if upper.startswith("APROBAR"):
        return "APROBAR"
    return "ESCALAR"


async def llm_classify(state: CaseState) -> CaseState:
    features = state["features"]
    rule_details = state.get("rule_details") or []
    senales_regla = state.get("senales_regla") or []
    es_forzado = state.get("decision_regla") == "ESCALAR"

    llm = _crear_cliente()

    contexto = (
        f"{features.get('num_compensaciones_90d', 0)} reclamos en 90d, "
        f"{features.get('flags_fraude_previos', 0)} flags fraude, "
        f"antiguedad {features.get('antiguedad_usuario_dias', 0)}d, "
        f"GPS {features.get('entrega_confirmada_gps', 'N/A')}, "
        f"comp_ratio {features.get('comp_ratio', 'N/A')}, "
        f"monto solicitado ${features.get('compensacion_solicitada_mxn', 0)}."
    )

    reglas_evaluadas_str = _formatear_reglas_evaluadas(rule_details)
    senales_str = "; ".join(senales_regla) if senales_regla else "ninguna"

    nota_forzado = (
        "\n\nNota: este caso fue PRE-MARCADO para escalación por palabras "
        "críticas de seguridad de marca. Tu análisis debe confirmar y justificar "
        "la escalación (riesgo legal/salud), no re-evaluar la decisión."
        if es_forzado
        else ""
    )

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        "--- CASO A ANALIZAR ---\n\n"
        f'Reclamo: "{features.get("descripcion_reclamo", "")}"\n\n'
        f"Contexto: {contexto}\n\n"
        f"Reglas evaluadas:\n{reglas_evaluadas_str}\n\n"
        f"Señales detectadas: {senales_str}\n"
        f"{nota_forzado}\n\n"
        "Responde SOLO con el JSON exacto sin markdown."
    )

    async def _invocar() -> str:
        async with LLM_SEMAFORO:
            respuesta = await llm.ainvoke(prompt)
        contenido = respuesta.content
        if isinstance(contenido, list):
            contenido = "".join(
                b.get("text", "") for b in contenido if isinstance(b, dict)
            )
        return str(contenido)

    try:
        analisis = None
        for _ in range(get_llm_config().intentos_parsing):
            contenido = await _invocar()
            analisis = _parsear_analisis(contenido)
            if analisis is not None:
                break
    except Exception as e:
        logger.warning("llm_classify: fallo LLM en %s: %s", state.get("case_id"), e)
        analisis = None

    if analisis is None:
        analisis = {
            "justificacion": "No se pudo interpretar la respuesta del modelo.",
            "resumen": "No se pudo generar el resumen automático.",
            "veredicto": "ESCALAR: revisión manual requerida (error de parsing)",
            "señales_explicadas": [],
            "error": "parsing",
        }

    # Normalizar señales a nombres canónicos snake_case.
    for s in analisis.get("señales_explicadas", []):
        if isinstance(s, dict) and s.get("señal"):
            s["señal"] = normalizar_señal_llm(str(s["señal"]))

    state["llm_analysis"] = analisis
    state["decision_llm"] = _veredicto_discreto(analisis.get("veredicto", ""))
    state["llm_resultado"] = {
        "resumen": analisis.get("resumen", ""),
        "veredicto": analisis.get("veredicto", ""),
        "señales_explicadas": analisis.get("señales_explicadas", []),
    }
    return state

## Pipeline: Nodos 5-6 + compilación del grafo

`final_decision` combina reglas + LLM:
- Reglas deciden primero (jerarquía determinista: RECHAZAR > APROBAR)
- Palabras críticas → ESCALAR (seguridad marca)
- LLM decide en ambiguos vía `veredicto`:
  - `APROBAR: ...` → APROBAR (jerarquía)
  - `RECHAZAR: ...` → RECHAZAR (jerarquía)
  - `ESCALAR: ...` o vacío → ESCALAR (seguridad marca)

`generate_output` formatea y persiste el checklist en `resolution_case.reglas_checklist`.

In [17]:
def _sin_prefijo(texto: str) -> str:
    """Quita el prefijo "APROBAR:" / "RECHAZAR:" / "ESCALAR:" de un texto."""
    t = (texto or "").strip()
    for p in ("APROBAR", "RECHAZAR", "ESCALAR"):
        if t.upper().startswith(p + ":"):
            return t[len(p) + 1:].strip()
    return t


_TIPO_POR_DECISION = {
    "APROBAR": "APROBAR",
    "RECHAZAR": "RECHAZAR",
    "ESCALAR": "ESCALAR_FORZOSO",
}


def _justificacion_reglas(rule_details: list[dict], decision: str) -> str:
    """Justificación de reglas: un bloque por regla disparada.

    Cada bloque tiene dos líneas (separadas por ``\n``):
        ``regla_id — descripcion``
        ``explicacion``  (texto preseteado; se omite si la regla no lo tiene)
    Los bloques se unen con ``\n\n``.
    """
    tipo = _TIPO_POR_DECISION.get(decision)
    disparadas = [
        r for r in (rule_details or [])
        if r.get("se_disparo") and r.get("tipo_regla") == tipo
    ]
    if not disparadas:
        disparadas = [r for r in (rule_details or []) if r.get("se_disparo")]
    if not disparadas:
        return ""
    bloques = []
    for r in disparadas:
        desc = r.get("descripcion") or r.get("nombre", "")
        cabecera = f"{r.get('regla_id', '?')} — {desc}" if desc else r.get("regla_id", "?")
        explicacion = (r.get("explicacion") or "").strip()
        bloques.append(f"{cabecera}\n{explicacion}" if explicacion else cabecera)
    return "\n\n".join(bloques)


def _llm_justificacion(llm: dict) -> str | None:
    """Justificación del LLM, sin prefijo de decisión (o None si no aplica)."""
    if not llm:
        return None
    veredicto = (llm.get("veredicto") or "").strip()
    just = _sin_prefijo(llm.get("justificacion", "")) or _sin_prefijo(veredicto)
    return just or None


def _llm_senales(llm: dict) -> list[str]:
    """Señales canónicas generadas por el LLM (snake_case)."""
    if not llm:
        return []
    return [
        s.get("señal", "") for s in llm.get("señales_explicadas", [])
        if isinstance(s, dict) and s.get("señal")
    ]


def final_decision(state: CaseState) -> CaseState:
    """Combina reglas + LLM. Tres vías:
    1. APROBAR/RECHAZAR por reglas → decisión fija, sin LLM.
    2. ESCALAR forzoso → decisión FORZADA a ESCALAR; el LLM aporta análisis.
    3. AMBIGUO → decide el LLM vía veredicto (decision_llm).
    """
    rule_result = state.get("rule_result")
    decision_regla = state.get("decision_regla", "AMBIGUO")
    senales_regla = [s for s in state.get("senales_regla", []) if s and s.strip()]
    llm = state.get("llm_analysis") or {}

    # 1. APROBAR / RECHAZAR por reglas (sin LLM)
    if rule_result in ("APROBAR", "RECHAZAR"):
        state["final_decision"] = rule_result
        state["decision_regla"] = decision_regla
        state["decision_llm"] = None
        state["justificacion_regla"] = _justificacion_reglas(
            state.get("rule_details", []), rule_result)
        state["justificacion_llm"] = None
        state["senales_regla"] = senales_regla
        state["senales_llm"] = []
        state["justification"] = state["justificacion_regla"]
        return state

    # 2. ESCALAR forzoso (palabras críticas): decisión forzada a ESCALAR
    if rule_result == "ESCALAR":
        state["final_decision"] = "ESCALAR"
        state["decision_regla"] = "ESCALAR"
        state["justificacion_regla"] = _justificacion_reglas(
            state.get("rule_details", []), "ESCALAR")
        state["justificacion_llm"] = _llm_justificacion(llm)
        state["senales_regla"] = senales_regla
        state["senales_llm"] = _llm_senales(llm)
        state["justification"] = (
            state["justificacion_llm"] or state["justificacion_regla"])
        return state

    # 3. AMBIGUO → decide el LLM vía veredicto
    decision_llm = state.get("decision_llm") or "ESCALAR"
    state["decision_llm"] = decision_llm
    state["final_decision"] = decision_llm
    state["justificacion_regla"] = ""
    state["justificacion_llm"] = _llm_justificacion(llm)
    state["senales_regla"] = []
    state["senales_llm"] = _llm_senales(llm)
    state["justification"] = state["justificacion_llm"] or (
        "Revisión manual requerida por ambigüedad en la evidencia.")
    state["decision_regla"] = decision_regla
    return state


def generate_output(state: CaseState) -> CaseState:
    """Formatea el output final y consolida ``reglas_checklist`` (JSONB)."""
    # Señales deduplicadas (regla y LLM por separado).
    state["senales_regla"] = list(dict.fromkeys(
        s for s in state.get("senales_regla", []) if s and s.strip()))
    state["senales_llm"] = list(dict.fromkeys(
        s for s in state.get("senales_llm", []) if s and s.strip()))

    # Asegurar los resultados discretos para la persistencia.
    state["decision_regla"] = state.get("decision_regla", "AMBIGUO")
    state["decision_llm"] = state.get("decision_llm")

    # Truncados: justificacion_regla es multilínea (1200); el resto 500.
    for campo, max_len in (
        ("justificacion_regla", 1200),
        ("justificacion_llm", 500),
        ("justification", 500),
    ):
        texto = state.get(campo, "") or ""
        if len(texto) > max_len:
            state[campo] = texto[:max_len - 3] + "..."
    if not state.get("llm_resultado"):
        state["llm_resultado"] = None

    # Consolidar el checklist por regla (anclado a su versión); lo persiste
    # services.persistir_decision en resolution_case.reglas_checklist.
    rule_details = state.get("rule_details") or []
    if rule_details:
        try:
        except Exception:
            state["reglas_checklist"] = [dict(r, version=0) for r in rule_details]
    else:
        state["reglas_checklist"] = []
    return state


def _route_after_rules(state: CaseState) -> str:
    # Solo APROBAR/RECHAZAR terminan directo; ESCALAR y AMBIGUO van al LLM.
    if state.get("rule_result") in ("APROBAR", "RECHAZAR"):
        return "final_decision"
    return "llm_classify"


def build_graph():
    """Compila el StateGraph con los nodos inline."""
    graph = StateGraph(CaseState)
    graph.add_node("load_case", load_case)
    graph.add_node("compute_features", compute_features)
    graph.add_node("apply_rules", apply_rules)
    graph.add_node("llm_classify", llm_classify)
    graph.add_node("final_decision", final_decision)
    graph.add_node("generate_output", generate_output)
    graph.set_entry_point("load_case")
    graph.add_edge("load_case", "compute_features")
    graph.add_edge("compute_features", "apply_rules")
    graph.add_conditional_edges("apply_rules", _route_after_rules,
        {"final_decision": "final_decision", "llm_classify": "llm_classify"})
    graph.add_edge("llm_classify", "final_decision")
    graph.add_edge("final_decision", "generate_output")
    graph.add_edge("generate_output", END)
    return graph.compile()


graph = build_graph()
print('[OK] Grafo compilado con nodos inline')

[OK] Grafo compilado con nodos inline


## Análisis individual

Cambia `CASO_ID` para analizar cualquier caso. Muestra decisión, justificación,
señales, checklist de reglas y análisis LLM (si aplica).

In [18]:
async def analizar_caso(caso_id: str, save: bool = False):
    """Analiza un caso individual y muestra el resultado detallado."""
    result = await graph.ainvoke({"case_id": caso_id})
    print(f"{'='*60}")
    print(f"Caso: {caso_id}")
    print(f"Decisión: {result['final_decision']} ({result.get('rule_disparada', 'sin regla')})")
    print(f"Reglas: {result.get('decision_regla', 'N/A')} | LLM: {result.get('decision_llm', '—')}")
    print(f"{'='*60}")
    if result.get('justificacion_regla'):
        print(f"\nJustificación (reglas): {result['justificacion_regla']}")
    if result.get('justificacion_llm'):
        print(f"\nJustificación (LLM): {result['justificacion_llm']}")
    if result.get('senales_regla'):
        print(f"\nSeñales (reglas): {result['senales_regla']}")
    if result.get('senales_llm'):
        print(f"\nSeñales (LLM): {result['senales_llm']}")
    if result.get('rule_details'):
        print(f"\n--- Checklist de reglas ---")
        for r in result['rule_details']:
            status = '✓' if r['se_disparo'] else '✗'
            print(f"  {status} {r['nombre']}: {r['detalle'][:120]}")
    if result.get('llm_analysis') and SHOW_LLM_DETAIL:
        llm_r = result['llm_analysis']
        print(f"\n--- Análisis LLM ---")
        print(f"  veredicto: {llm_r.get('veredicto', 'N/A')}")
        if result.get('llm_resultado'):
            lr = result['llm_resultado']
            print(f"  resumen: {lr.get('resumen', 'N/A')}")
            for s in lr.get('señales_explicadas', []):
                print(f"    • {s.get('señal','')}: {s.get('explicacion','')} (peso: {s.get('peso','')})")
    if save and SAVE_TO_DB:
        _persistir_caso(caso_id, result)
        print(f"\n[OK] Persistido en DB")
    return result

# Ejemplo: caso ambiguo que va al LLM
await analizar_caso('COMP-0001')


Caso: COMP-0001
Decisión: APROBAR (sin regla)
Reglas: AMBIGUO | LLM: APROBAR

Justificación (LLM): Usuario con antigüedad superior a 4 años, 0 flags de fraude y baja frecuencia de reclamos. El motivo es operativo estándar y no presenta inconsistencias ni palabras críticas.

Señales (LLM): ['descripcion_incoherente']

--- Checklist de reglas ---
  ✗ flags_fraude_previos >= 2: flags_fraude_previos >= 2 no se disparó: flags_fraude_previos=0 (>= 2) ✗
  ✗ comp_ratio > 0.99: comp_ratio > 0.99 no se disparó: comp_ratio=0.6336 (> 0.99) ✗
  ✗ freq_densidad > 0.66: freq_densidad > 0.66 no se disparó: freq_densidad=0.022222 (> 0.66) ✗
  ✗ compensacion_solicitada_mxn > 604.64: compensacion_solicitada_mxn > 604.64 no se disparó: compensacion_solicitada_mxn=423.76 (> 604.64) ✗
  ✗ flag_inconsistencia_gps == True: flag_inconsistencia_gps == True no se disparó: flag_inconsistencia_gps=False (== True) ✗
  ✗ flag_account_abuse == True: flag_account_abuse == True no se disparó: flag_account_abuse=False (

{'case_id': 'COMP-0001',
 'raw_data': {'caso_id': 'COMP-0001',
  'usuario_id': 'USR-11567',
  'antiguedad_usuario_dias': 1590,
  'ciudad': 'CDMX',
  'vertical': 'Comida',
  'restaurante': 'La Cocina de Doña Rosa',
  'valor_orden_mxn': Decimal('668.79'),
  'compensacion_solicitada_mxn': Decimal('423.76'),
  'num_compensaciones_90d': 2,
  'monto_compensado_90d_mxn': Decimal('103.53'),
  'entrega_confirmada_gps': 'NO confirmada',
  'tiempo_entrega_real_min': 56,
  'flags_fraude_previos': 0,
  'motivo_reclamo': 'Producto incorrecto',
  'descripcion_reclamo': 'Llegó comida diferente, creo que confundieron mi pedido.',
  'recomendacion_agente': 'ESCALAR',
  'es_sintetico': False},
 'features': {'caso_id': 'COMP-0001',
  'usuario_id': 'USR-11567',
  'antiguedad_usuario_dias': 1590,
  'ciudad': 'CDMX',
  'vertical': 'Comida',
  'restaurante': 'La Cocina de Doña Rosa',
  'valor_orden_mxn': Decimal('668.79'),
  'compensacion_solicitada_mxn': Decimal('423.76'),
  'num_compensaciones_90d': 2,
  'm

## Batch processing con control de sobreescritura

- `OVERWRITE=False`: solo procesa casos sin entrada en `resolution_case` (pendientes)
- `OVERWRITE=True`: reprocesa todos los casos
- `SAVE_TO_DB=True`: persiste en PostgreSQL (`resolution_case`)
- `SAVE_TO_DB=False`: solo output en memoria

In [ ]:
from src.utils.jsonb import jsonb
def _persistir_caso(caso_id: str, resultado: dict):
    """Persiste la decisión del pipeline en resolution_case (contrato vigente)."""
    decision_regla = resultado.get("decision_regla")
    llm_res = resultado.get("llm_resultado")
    # ESCALAR forzado por reglas: la decisión es de las reglas aunque el LLM
    # haya generado análisis enriquecido (justificación/señales).
    fuente = "reglas" if decision_regla == "ESCALAR" else ("llm" if llm_res else "reglas")
    features_version = resultado.get("features_version", "v1")
    conn = db_conectar()
    try:
        with conn.cursor() as cur:
            cur.execute(
                """INSERT INTO resolution_case
                       (caso_id, features_version, fuente, decision,
                        decision_regla, reglas_checklist,
                        decision_llm, justificacion_llm, justificacion_regla,
                        senales_llm, senales_regla, llm_resultado)
                   VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                   ON CONFLICT (caso_id) DO UPDATE SET
                       features_version    = EXCLUDED.features_version,
                       fuente              = EXCLUDED.fuente,
                       decision            = EXCLUDED.decision,
                       decision_regla      = EXCLUDED.decision_regla,
                       reglas_checklist    = EXCLUDED.reglas_checklist,
                       decision_llm        = EXCLUDED.decision_llm,
                       justificacion_llm   = EXCLUDED.justificacion_llm,
                       justificacion_regla = EXCLUDED.justificacion_regla,
                       senales_llm         = EXCLUDED.senales_llm,
                       senales_regla       = EXCLUDED.senales_regla,
                       llm_resultado       = EXCLUDED.llm_resultado,
                       updated_at          = NOW()""",
                (
                    caso_id,
                    features_version,
                    fuente,
                    resultado["final_decision"],
                    decision_regla or "AMBIGUO",
                    jsonb(resultado.get("reglas_checklist") or []),
                    resultado.get("decision_llm"),
                    resultado.get("justificacion_llm"),
                    resultado.get("justificacion_regla") or None,
                    " | ".join(resultado.get("senales_llm") or []),
                    " | ".join(resultado.get("senales_regla") or []),
                    jsonb(llm_res),
                ),
            )
        conn.commit()
    finally:
        conn.close()

import asyncio

async def _invocar_grafo(cid: str):
    """Ejecuta el grafo async en el event loop del kernel."""
    return await graph.ainvoke({"case_id": cid})

async def procesar_batch(overwrite=OVERWRITE, save_to_db=SAVE_TO_DB, max_paralelo=None):
    """Procesa casos en batch con control de sobreescritura y ejecución paralela.

    La concurrencia está acotada por ``max_paralelo`` (default: el mismo
    ``max_concurrencia`` del LLM en ``src/config/model.yaml``). El nodo
    ``llm_classify`` ya limita las llamadas al proveedor vía su semáforo
    interno; aquí se limita también el trabajo de DB/grafo en paralelo.
    """
    if max_paralelo is None:
        from src.config import get_llm_config
        max_paralelo = get_llm_config().max_concurrencia
    conn = db_conectar()
    try:
        with conn.cursor() as cur:
            if overwrite:
                cur.execute("SELECT caso_id FROM cases ORDER BY caso_id")
            else:
                cur.execute(
                    """SELECT c.caso_id FROM cases c
                       LEFT JOIN resolution_case a ON a.caso_id = c.caso_id
                       WHERE a.caso_id IS NULL
                       ORDER BY c.caso_id"""
                )
            case_ids = [r[0] for r in cur.fetchall()]
    finally:
        conn.close()
    if not case_ids:
        print("[INFO] No hay casos para procesar.")
        return pd.DataFrame()
    print(f"Procesando {len(case_ids)} casos (overwrite={overwrite}, save_to_db={save_to_db}, max_paralelo={max_paralelo})...")
    resultados = []
    stats = {"APROBAR": 0, "RECHAZAR": 0, "ESCALAR": 0}
    t0 = time.time()
    sem = asyncio.Semaphore(max_paralelo)

    async def _procesar_uno(cid):
        async with sem:
            try:
                result = await _invocar_grafo(cid)
                if save_to_db:
                    _persistir_caso(cid, result)
                return cid, result, None
            except Exception as e:
                return cid, None, e

    tareas = [_procesar_uno(cid) for cid in case_ids]
    completados = 0
    for fut in asyncio.as_completed(tareas):
        cid, result, error = await fut
        completados += 1
        if error is not None:
            print(f"  [ERROR] {cid}: {error}")
            stats["ESCALAR"] += 1
            continue
        resultados.append({
            "caso_id": cid,
            "decision": result["final_decision"],
            "regla": result.get("rule_disparada", ""),
            "justificacion_regla": (result.get("justificacion_regla") or "")[:100],
            "justificacion_llm": (result.get("justificacion_llm") or "")[:100],
            "senales_regla": " | ".join(result.get("senales_regla", [])),
            "senales_llm": " | ".join(result.get("senales_llm", [])),
            "llm_usado": result.get("llm_analysis") is not None,
        })
        stats[result["final_decision"]] += 1
        if completados % 10 == 0 or completados == len(case_ids):
            elapsed = time.time() - t0
            eta = (elapsed / completados) * (len(case_ids) - completados) if completados > 0 else 0
            print(f"  [{completados}/{len(case_ids)}] {cid}: {result['final_decision']} (ETA: {eta:.0f}s)")
    print(f"\nCompletado en {time.time() - t0:.0f}s")
    print(f"Distribución: {stats}")
    return pd.DataFrame(resultados)

# Mostrar estado actual
conn = db_conectar()
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM cases c LEFT JOIN resolution_case a ON a.caso_id = c.caso_id WHERE a.caso_id IS NULL")
        pendientes = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM cases")
        total = cur.fetchone()[0]
    conn.close()
except:
    pendientes = total = 0
print(f"Casos en DB: {total} | Pendientes (sin analizar): {pendientes}")
print(f"OVERWRITE={OVERWRITE} | SAVE_TO_DB={SAVE_TO_DB}")
print("Descomenta procesar_batch() arriba para ejecutar.")
df_resultados = await procesar_batch()
print(df_resultados.head(10).to_string(index=False))


## Visualización de resultados

Distribución de decisiones, señales más frecuentes y reglas disparadas.

In [20]:
import matplotlib.pyplot as plt

# Consultar distribución actual de la DB
conn = db_conectar()
try:
    with conn.cursor() as cur:
        cur.execute("SELECT decision, COUNT(*) AS n FROM resolution_case GROUP BY decision ORDER BY n DESC")
        dist_rows = cur.fetchall()
        cur.execute("SELECT c.es_sintetico, a.decision, COUNT(*) AS n FROM cases c JOIN resolution_case a ON a.caso_id = c.caso_id GROUP BY c.es_sintetico, a.decision ORDER BY 1, 3 DESC")
        origen_rows = cur.fetchall()
        cur.execute("SELECT fuente, decision, COUNT(*) AS n FROM resolution_case GROUP BY fuente, decision ORDER BY fuente, decision")
        conf_rows = cur.fetchall()
    conn.close()
except:
    dist_rows = origen_rows = conf_rows = []

# Gráficos
if dist_rows:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = {'APROBAR': '#4CAF50', 'RECHAZAR': '#F44336', 'ESCALAR': '#FF9800'}

    # 1. Distribución
    ax = axes[0]
    labels = [r[0] for r in dist_rows]
    values = [r[1] for r in dist_rows]
    ax.bar(labels, values, color=[colors.get(x, '#999') for x in labels])
    ax.set_title('Distribución de decisiones')
    ax.set_ylabel('Casos')
    for j, v in enumerate(values):
        ax.text(j, v + 1, str(v), ha='center')

    # 2. Por origen
    ax = axes[1]
    if origen_rows:
        df_orig = pd.DataFrame(origen_rows, columns=['sintetico', 'decision', 'n'])
        pivot = df_orig.pivot(index='decision', columns='sintetico', values='n').fillna(0)
        pivot.plot(kind='bar', ax=ax, color=['#2196F3', '#FF9800'])
        ax.set_title('Por origen (original vs sintético)')
        ax.set_ylabel('Casos')
        ax.set_xlabel('')
        ax.legend(['Original', 'Sintético'])
        ax.tick_params(axis='x', rotation=0)

    # 3. Decisiones por fuente (reglas vs LLM)
    ax = axes[2]
    if conf_rows:
        df_conf = pd.DataFrame(conf_rows, columns=['fuente', 'decision', 'n'])
        pivot_c = df_conf.pivot(index='decision', columns='fuente', values='n').fillna(0)
        pivot_c.plot(kind='bar', ax=ax, color=['#4CAF50', '#9E9E9E'])
        ax.set_title('Decisiones por fuente (reglas vs LLM)')
        ax.set_ylabel('Casos')
        ax.set_xlabel('')
        ax.legend(pivot_c.columns)
        ax.tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()
else:
    print('[INFO] DB no disponible para visualización.')

[INFO] DB no disponible para visualización.


## Análisis de casos con LLM

Casos donde el LLM produjo análisis estructurado (resumen, veredicto, señales).

In [21]:
conn = db_conectar()
try:
    with conn.cursor() as cur:
        cur.execute("SELECT c.caso_id, a.decision, a.fuente, a.llm_resultado FROM cases c JOIN resolution_case a ON a.caso_id = c.caso_id WHERE a.llm_resultado IS NOT NULL ORDER BY c.caso_id")
        rows = cur.fetchall()
    conn.close()
except:
    rows = []

if rows:
    df_llm = pd.DataFrame(rows, columns=['caso_id', 'decision', 'fuente', 'llm_resultado'])
    print(f'Casos con análisis LLM: {len(df_llm)}')
    print(f'\nDistribución de decisiones (casos LLM):')
    print(df_llm['decision'].value_counts().to_string())
    print(f'\n--- Ejemplos de análisis LLM ---')
    for _, r in df_llm.head(2).iterrows():
        print(f'\nCaso {r["caso_id"]}: {r["decision"]} (fuente: {r.get("fuente", "")})')
        lr = r['llm_resultado']
        if lr:
            print(f'  resumen: {lr.get("resumen", "N/A")}')
            print(f'  veredicto: {lr.get("veredicto", "N/A")}')
            for s in lr.get('señales_explicadas', []):
                print(f'    • {s.get("señal", "")}: {s.get("explicacion", "")} (peso: {s.get("peso", "")})')
else:
    print('[INFO] No hay casos con análisis LLM en la DB.')
    print('Ejecuta procesar_batch() con OVERWRITE=True para generarlos.')

[INFO] No hay casos con análisis LLM en la DB.
Ejecuta procesar_batch() con OVERWRITE=True para generarlos.


## Verificación de integridad

Confirma que todos los casos procesados tienen decisión, justificación y señales.

In [22]:
conn = db_conectar()
try:
    with conn.cursor() as cur:
        cur.execute("""SELECT COUNT(*) AS total,
                          COUNT(a.decision) AS con_decision,
                          COUNT(a.justificacion_regla) + COUNT(a.justificacion_llm) AS con_just,
                          COUNT(a.senales_regla) + COUNT(a.senales_llm) AS con_senales
                   FROM cases c LEFT JOIN resolution_case a ON a.caso_id = c.caso_id
        """)
        row = cur.fetchone()
    conn.close()
except:
    row = (0, 0, 0, 0)

total, con_dec, con_just, con_senales = row
print(f'Total casos:          {total}')
print(f'Con decisión:         {con_dec}')
print(f'Con justificación:    {con_just}')
print(f'Con señales:          {con_senales}')
print()
if total > 0:
    assert con_dec == total, f'{total - con_dec} casos sin decisión'
    assert con_just >= total, f'{total - con_just} casos sin justificación'
    assert con_senales >= total, f'{total - con_senales} casos sin señales'
    print('[OK] Verificación superada: todos los casos tienen decisión + justificación + señales.')
else:
    print('[SKIP] DB no disponible.')


Total casos:          250
Con decisión:         0
Con justificación:    0
Con señales:          0



AssertionError: 250 casos sin decisión

In [ ]:
# ── Entregable: exportar Excel vía API + batch demo en memoria ──
import io
import httpx

API = os.getenv("API_URL", "http://localhost:8000")
client = httpx.Client(timeout=300)

# Export de los 150 casos originales (mismo endpoint que usa la UI)
resp = client.get(f"{API}/export/excel", params={"es_sintetico": False})
resp.raise_for_status()
out = PROYECTO / "data" / "150casos_analizados.xlsx"
out.parent.mkdir(exist_ok=True)
out.write_bytes(resp.content)
df = pd.read_excel(io.BytesIO(resp.content))
print(f"[OK] Excel de {len(df)} casos → {out}")
print(df["recomendacion"].value_counts().to_string())

# Batch demo en memoria (persistir=False: NO escribe resolution_case)
resp = client.post(f"{API}/analyze/batch", json={
    "case_ids": ["COMP-0002", "COMP-0006", "COMP-0009"],
    "persistir": False,
})
resp.raise_for_status()
job_id = resp.json()["job_id"]
print(f"[OK] Batch demo lanzado (job {job_id})…")
while True:
    estado = client.get(f"{API}/jobs/{job_id}").json()
    if estado["status"] == "done":
        break
    time.sleep(2)
print(f"[OK] Distribución demo: {estado["decisiones"]}")

demo = client.get(f"{API}/jobs/{job_id}/excel")
demo.raise_for_status()
out_demo = PROYECTO / "data" / "batch_demo_resultados.xlsx"
out_demo.write_bytes(demo.content)
print(f"[OK] Excel de la demo → {out_demo}")

# Verificación: la demo no escribió en resolution_case
conn = db_conectar()
try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM resolution_case")
        print(f"[INFO] resolution_case: {cur.fetchone()[0]} filas (la demo no escribe aquí)")
finally:
    conn.close()
client.close()


## Conclusiones

- **Pipeline inline:** el código de los 6 nodos está en esta notebook, editable en caliente.
- **Flags interactivos:** `OVERWRITE`, `SAVE_TO_DB`, `USE_DB_RULES` controlan el comportamiento.
- **Batch con control:** `procesar_batch()` salta casos ya analizados si `OVERWRITE=False`.
- **Análisis individual:** `await analizar_caso('COMP-XXXX')` muestra el detalle completo.
- **Verificación:** assertions confirman que todos los casos tienen decisión + justificación + señales.

> **Nota:** Para reprocesar todos los casos, cambiar `OVERWRITE = True` en la celda de
> Setup y ejecutar `await procesar_batch()`. El LLM puede tardar ~15s por caso ambiguo (async).